# Tráfico en Madrid: Preparación del Dataset

Este es el notebook que convierte los datos brutos en un dataset listo para entrenar un modelo. Es menos visual que el EDA y más mecánico, pero es probablemente el notebook donde más decisiones importantes se toman. La calidad del modelo final depende mucho más de cómo preparemos los datos aquí que de cuántos árboles tenga el Random Forest en el notebook 03.

El plan es el siguiente. Cargamos los mismos datos que en el EDA, les quitamos las filas con errores de sensor, las unimos con la información geográfica, construimos las features (hora, día de la semana, distrito, etc.) que va a consumir el modelo, definimos la variable objetivo binaria aplicando el umbral que justificamos en el EDA, verificamos que no haya data leakage, codificamos las variables categóricas, partimos los datos en train y test, y guardamos todo en disco para que los notebooks 03 y 04 puedan cargarlo directamente sin volver a ejecutar este pipeline.

## Configuración del entorno

Mismos imports que el notebook 01 más algunas piezas específicas para esta fase.

In [ ]:
import pickle
import zipfile
from pathlib import Path

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split


`pickle` lo usaremos al final para guardar los splits y el DataFrame de sensores en un formato que preserva los dtypes exactos de pandas. `train_test_split` es la función estándar de scikit-learn para partir un dataset en entrenamiento y test.

Las rutas: la novedad respecto al notebook 01 es `DATA_PROCESSED`, que es donde vamos a dejar el dataset preparado.

In [ ]:
DATA_RAW       = Path('../data/raw')
DATA_HIST      = DATA_RAW / 'historico'
DATA_SENS      = DATA_RAW / 'sensores'
DATA_PROCESSED = Path('../data/processed')
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)


Las constantes del proyecto. Las tres son decisiones que ya tomamos o que son convenciones que mantenemos consistentes.

In [ ]:
RANDOM_STATE = 42        # Reproducibilidad: cualquier operación aleatoria usa esta semilla
UMBRAL_CARGA = 70        # Decidido en el EDA a partir de la CDF de carga
TEST_SIZE    = 0.20      # Partición clásica 80/20


Las URLs las repetimos aquí para que el notebook sea autónomo. Si alguien ejecuta directamente el 02 sin haber pasado por el 01, los datos se descargan igualmente. De momento seguimos trabajando con marzo de 2026.

In [ ]:
URLS_HISTORICO = {
    '2026-03': 'https://datos.madrid.es/egob/catalogo/208627-218-transporte-ptomedida-historico/download/03-2026.zip',
}
URL_SENSORES = 'https://datos.madrid.es/egob/catalogo/202468-0-intensidad-trafico.csv'
MESES        = ['2026-03']


Configuración estética, idéntica a la del notebook anterior, para que las gráficas que salgan aquí casen con el resto.

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

sns.set_theme(style='ticks', palette='muted', font_scale=0.9)
plt.rcParams.update({
    'figure.figsize': (10, 4), 'figure.dpi': 100,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titleweight': 'normal', 'axes.titlelocation': 'left',
    'axes.titlesize': 11, 'axes.labelsize': 10,
    'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'legend.frameon': False, 'axes.grid': True,
    'grid.linestyle': ':', 'grid.alpha': 0.35,
})

COLOR_PRIMARIO   = '#4C72B0'
COLOR_SECUNDARIO = '#DD8452'


## 1. Carga de los datos crudos

El notebook tiene que funcionar tanto si los ficheros están ya en disco (porque el 01 los descargó) como si no. Así que primero nos aseguramos de tenerlos, y luego cargamos.

In [ ]:
def descargar_si_falta(url: str, destino: Path) -> None:
    # Clave para reejecuciones rápidas: si ya lo tenemos, no hacemos nada.
    if destino.exists():
        return
    destino.parent.mkdir(parents=True, exist_ok=True)
    respuesta = requests.get(url, stream=True)
    respuesta.raise_for_status()
    with open(destino, 'wb') as f:
        for chunk in respuesta.iter_content(chunk_size=1024 * 1024):
            f.write(chunk)
    print(f'  ↓ descargado {destino.name}')


Función idéntica en espíritu a la del notebook 01, con un pequeño cambio: el `if destino.exists()` va primero sin imprimir nada, para no contaminar la salida cuando todo ya está en disco.

In [ ]:
# Nos aseguramos de que todo está descargado
for mes in MESES:
    descargar_si_falta(URLS_HISTORICO[mes], DATA_HIST / f'{mes}.zip')
descargar_si_falta(URL_SENSORES, DATA_SENS / 'sensores.csv')


La función de carga del ZIP es exactamente la misma del notebook 01. La duplicamos aquí en vez de importarla para mantener los notebooks desacoplados. En un proyecto más grande tendría sentido moverla a un módulo Python compartido, pero para cuatro notebooks el esfuerzo no compensa.

In [ ]:
def cargar_mes(ruta_zip: Path) -> pd.DataFrame:
    with zipfile.ZipFile(ruta_zip) as z:
        nombre_csv = z.namelist()[0]
        with z.open(nombre_csv) as f:
            return pd.read_csv(f, sep=';', parse_dates=['fecha'])


In [ ]:
df_historico = pd.concat(
    [cargar_mes(DATA_HIST / f'{m}.zip') for m in MESES],
    ignore_index=True,
)
df_sensores = pd.read_csv(DATA_SENS / 'sensores.csv', sep=';')

print(f'Histórico: {df_historico.shape[0]:,} filas')
print(f'Sensores:  {df_sensores.shape[0]:,} puntos de medida')


## 2. Tratamiento de errores de sensor

En el EDA vimos que hay dos marcadores de error conviviendo en el dataset. Por un lado, `carga == -1`: un valor imposible en una variable que por definición va de 0 a 100, y que el Ayuntamiento usa como código numérico de fallo. Por otro, la columna `error` con valor `'S'`, que marca explícitamente lecturas defectuosas. Los dos se solapan bastante pero no al 100%, así que vamos a eliminar cualquier fila que tenga alguno de los dos.

¿Por qué eliminar y no imputar? Porque imputar tiene sentido cuando el dato falta por razones aleatorias y tenemos información para reconstruirlo. Aquí, cuando un sensor marca error, no sabemos si el tráfico real era fluido o atascado — simplemente no hay dato. Inventárselo introduciría ruido sin ganar nada. Y además, tenemos millones de lecturas buenas: perder un pequeño porcentaje no compromete nada.

In [ ]:
tam_antes = len(df_historico)

# Máscara booleana con las filas que SÍ queremos conservar.
# Una lectura es válida si tiene carga >= 0 Y no está marcada como error.
mascara_valida = (df_historico['carga'] != -1) & (df_historico['error'] != 'S')
df_limpio = df_historico[mascara_valida].copy()

tam_despues = len(df_limpio)
perdidos    = tam_antes - tam_despues


Hecho. Imprimimos el balance: cuántas filas teníamos, cuántas nos quedan, cuántas hemos tirado y qué proporción supone.

In [ ]:
print(f'Registros antes:   {tam_antes:>12,}')
print(f'Registros después: {tam_despues:>12,}')
print(f'Eliminados:        {perdidos:>12,}  ({perdidos / tam_antes:.2%})')


Perdemos un porcentaje pequeño, coherente con lo que anticipamos en el EDA. El grueso del dataset sobrevive intacto y ya no tenemos que preocuparnos por valores físicamente imposibles ni por lecturas que el propio sensor marcó como erróneas.

## 3. Merge con los metadatos de los sensores

El histórico nos dice **qué** midió cada sensor, pero no **dónde** está. Para que el modelo pueda usar información geográfica (distrito, coordenadas) necesitamos combinar los dos datasets. El hilo común es la columna `id`, que identifica unívocamente cada punto de medida.

Antes de unir, nos quedamos solo con las columnas del catálogo de sensores que son útiles. No tiene sentido arrastrar columnas que no vamos a usar: ocupan memoria y entorpecen la exploración.

In [ ]:
# Columnas del catálogo que nos interesan para el modelo
cols_sensor = ['id', 'nombre', 'distrito', 'tipo_elem', 'utm_x', 'utm_y']

# drop_duplicates por si el catálogo tuviera sensores repetidos (defensivo)
df_sensores_mini = df_sensores[cols_sensor].drop_duplicates(subset='id')

print(f'Sensores únicos en el catálogo: {len(df_sensores_mini):,}')


Ahora el merge. Usamos un `left join` desde las mediciones: eso significa que conservamos todas las filas del histórico, y si un sensor no aparece en el catálogo, sus columnas de metadata quedarán a NaN. Es la elección correcta aquí: no queremos perder mediciones por culpa de que falte un registro en el catálogo.

In [ ]:
df = df_limpio.merge(df_sensores_mini, on='id', how='left', suffixes=('', '_sens'))
print(f'Dataset tras el merge: {len(df):,} filas × {df.shape[1]} columnas')


Comprobamos cuántos sensores del histórico no tienen metadata. Si el número es muy pequeño, los eliminamos y seguimos. Si fuera alto, habría que entender por qué — podría ser que el catálogo no esté actualizado respecto al histórico, o que haya sensores de prueba sin geolocalizar.

In [ ]:
sin_metadata = df['distrito'].isna().sum()
print(f'Registros sin metadata de sensor: {sin_metadata:,}  ({sin_metadata / len(df):.2%})')

if sin_metadata > 0:
    df = df.dropna(subset=['distrito']).copy()
    print(f'Eliminados. Tamaño final: {len(df):,}')


## 4. Feature engineering

Esta es la sección donde el dataset deja de ser una lista de lecturas y se convierte en las variables que va a consumir el modelo. Los modelos no aprenden nada útil del timestamp en crudo — necesitamos descomponerlo en señales que capturen los patrones temporales que vimos en el EDA.

### Extracción de componentes temporales

La `fecha` es un datetime. Pandas nos deja extraer cualquier componente con el accessor `.dt`: hora, día de la semana, mes, día del mes, etc. Extraemos las cuatro que nos interesan.

In [ ]:
df['hora']        = df['fecha'].dt.hour             # 0–23
df['dia_semana']  = df['fecha'].dt.dayofweek        # 0=lunes, 6=domingo
df['mes']         = df['fecha'].dt.month            # 1–12
df['dia_mes']     = df['fecha'].dt.day              # 1–31


Al convertir la fecha a estos cuatro enteros, perdemos información (por ejemplo, el año) pero ganamos algo más valioso: patrones comparables entre fechas. El modelo puede aprender "a las 8 de la mañana suele haber atasco" independientemente de si hablamos del 3 de marzo o del 24 de marzo.

### Features binarias derivadas

Aunque el modelo podría en teoría aprender "los laborables tienen más tráfico" solo con `dia_semana`, servirle esa información directamente como una feature binaria acelera el aprendizaje y reduce la varianza. Es la primera de las features derivadas: `es_laborable`.

In [ ]:
df['es_laborable'] = (df['dia_semana'] < 5).astype(int)


La segunda es `es_hora_punta`. Combinamos dos criterios: que sea día laborable y que la hora caiga en una de las dos franjas punta que identificamos en el EDA (7-9 y 17-20). Es una feature muy potente porque condensa una interacción: el modelo lineal no habría aprendido solo la combinación "laborable + hora punta" sin que se la diéramos explícitamente.

In [ ]:
# Operamos con los valores como booleanos y luego pasamos a int para coherencia de dtypes
df['es_hora_punta'] = (
    df['es_laborable'].astype(bool) &
    (df['hora'].between(7, 9) | df['hora'].between(17, 20))
).astype(int)

print('Proporción de lecturas en hora punta:', df['es_hora_punta'].mean().round(3))


### Franjas horarias

La tercera feature derivada agrupa las 24 horas en cinco categorías gruesas. La motivación es que el comportamiento del tráfico no cambia minuto a minuto sino por franjas (madrugada, mañana, mediodía, tarde, noche). El modelo podría inferirlo del campo `hora`, pero volver a darle la información agrupada no sobra — sobre todo para la regresión logística, que va a tratar `hora` como una variable lineal (o sea, asumiendo que el efecto de pasar de las 7 a las 8 es el mismo que de las 14 a las 15, lo cual es falso).

In [ ]:
def franja_horaria(h: int) -> str:
    if   h < 6:  return 'madrugada'
    elif h < 12: return 'mañana'
    elif h < 15: return 'mediodia'
    elif h < 21: return 'tarde'
    else:        return 'noche'

df['franja'] = df['hora'].apply(franja_horaria)

# Comprobación rápida de la distribución
df['franja'].value_counts()


Una última ojeada antes de seguir: verificamos las nuevas columnas en unas cuantas filas para confirmar que todo cuadra.

In [ ]:
df[['fecha', 'hora', 'dia_semana', 'mes', 'es_laborable', 'es_hora_punta', 'franja']].head(10)


## 5. Definición del target binario

Aquí traducimos la decisión del EDA en una columna nueva. La variable `carga` es continua de 0 a 100; nosotros queremos un problema de clasificación binaria, así que cortamos por el umbral del 70%: todo lo que lo iguala o supera es congestión.

In [ ]:
df['congestionado'] = (df['carga'] >= UMBRAL_CARGA).astype(int)

distribucion = df['congestionado'].value_counts(normalize=True).sort_index()
print(f'Fluido        (0): {distribucion[0]:.2%}')
print(f'Congestionado (1): {distribucion[1]:.2%}')


### ¿Es manejable este desbalance?

Una pregunta legítima: con solo un 4-6% de positivos, ¿no tenemos un problema serio de desbalance? Respuesta corta: está en el límite, pero es manejable. Un modelo entrenado con un desbalance así y sin ningún ajuste tiende a predecir siempre la clase mayoritaria (nunca congestión) porque así consigue un 94-96% de accuracy "gratis". Pero eso lo resolvemos de dos formas durante el entrenamiento: primero, usando métricas que castigan predecir siempre lo mismo (F1, recall, AUC en lugar de accuracy), y segundo, activando `class_weight='balanced'` en los modelos, que reequilibra internamente los pesos de cada clase. No necesitamos recurrir a técnicas más agresivas como SMOTE (que genera ejemplos sintéticos de la clase minoritaria) porque tenemos suficientes positivos en términos absolutos — cientos de miles, no decenas.

### Visualización del desbalance

Una gráfica rápida que ayuda a comunicarlo cuando se presenta el trabajo.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
conteo = df['congestionado'].value_counts().sort_index()
ax.barh(['Fluido', 'Congestionado'], conteo.values, color=[COLOR_PRIMARIO, COLOR_SECUNDARIO])
for i, v in enumerate(conteo.values):
    ax.text(v, i, f' {v:,} ({v/conteo.sum():.1%})', va='center')
ax.set_xlabel('Nº de registros')
ax.set_title(f'Distribución del target binario (umbral = {UMBRAL_CARGA}%)')
plt.tight_layout()
plt.show()


## 6. Selección de features y verificación de data leakage

Esta es la sección más delicada del notebook. **Data leakage** significa dejar que al modelo le llegue información que no estaría disponible en el momento real de hacer la predicción. Cuando ocurre, las métricas de validación se disparan (el modelo "clava" las predicciones) pero en producción todo se derrumba. Es uno de los errores más comunes y más silenciosos en machine learning aplicado.

En nuestro caso concreto, la trampa está en las columnas `intensidad`, `ocupacion` y `vmed`. Están fuertemente correlacionadas con `carga` — lo vimos en el EDA — pero eso no es lo malo. Lo malo es que **se miden en el mismo instante que `carga`**. Si el modelo las conoce, no está prediciendo el tráfico: está reconstruyendo el presente a partir del presente.

La regla para evitar data leakage es sencilla de enunciar pero fácil de olvidar: **al modelo solo puede llegar información que un usuario real podría conocer por adelantado**. En nuestro caso, el usuario dice "quiero saber si a las 8 de la mañana del próximo lunes la M-30 va a estar atascada". Información disponible: hora, día de la semana, mes, identificador del sensor, distrito, coordenadas. Información no disponible: lo que vaya a medir ese sensor en ese momento.

In [ ]:
# Variables que filtran información del target o son el propio target
columnas_prohibidas = {
    'carga',               # Es el propio target (continuo)
    'intensidad',          # Se mide simultáneamente con carga
    'ocupacion',           # Se mide simultáneamente con carga
    'vmed',                # Velocidad media, también simultánea
    'congestionado',       # El target binario
    'fecha',               # No se usa directamente; sus componentes (hora, dow, mes) sí
    'error',               # Ya filtrado, no aporta
    'periodo_integracion', # Metadato técnico del sensor, sin valor predictivo
    'es_error',            # Auxiliar creado en el EDA
}


Y aquí listamos explícitamente lo que SÍ entra al modelo. Mejor listar lo que queremos que lo que no queremos: así nada se cuela por descuido.

In [ ]:
# Features numéricas: todas son cosas calculables a partir del timestamp o de la posición
features_numericas   = [
    'hora',           # 0-23
    'dia_semana',     # 0-6
    'mes',            # 1-12
    'es_laborable',   # 0/1
    'es_hora_punta',  # 0/1
    'utm_x',          # coordenada X del sensor (metros)
    'utm_y',          # coordenada Y del sensor (metros)
]

# Features categóricas: las codificaremos a continuación
features_categoricas = ['distrito', 'franja', 'tipo_elem', 'id']

target = 'congestionado'


Dejamos un `assert` explícito para que si en el futuro alguien mueve cosas y accidentalmente cuela una columna prohibida, el notebook falle ruidosamente en lugar de seguir adelante con un modelo defectuoso. Es un guardrail barato que paga por sí mismo.

In [ ]:
assert not (set(features_numericas) | set(features_categoricas)) & columnas_prohibidas, \
    'Hay una feature filtrando información del target. Revisar columnas_prohibidas.'

print('Features numéricas:  ', features_numericas)
print('Features categóricas:', features_categoricas)
print('Target:              ', target)


Dos detalles que merecen un comentario. Incluimos `id` (identificador del sensor) como feature categórica. Esto le permite al modelo aprender comportamientos idiosincráticos de cada punto: una glorieta que siempre se colapsa frente a una calle residencial que nunca lo hace. Con miles de sensores y suficientes lecturas por sensor, esta información es muy potente; sin ella el modelo tendría que inferir la "personalidad" del sensor solo desde sus coordenadas y distrito, que es información más gruesa.

Y por otro lado, no incluimos `nombre` (nombre de la calle). El nombre es simplemente una etiqueta humana: "Castellana esquina Manuel Becerra" no aporta ninguna señal que no esté ya en `utm_x`, `utm_y` y `distrito`. Lo conservaremos en el DataFrame de sensores aparte para usarlo en la demo del notebook 04 — cuando presentemos una predicción al usuario, queremos poder mostrarle el nombre de la calle, no un número de sensor.

## 7. Codificación de variables categóricas

Los modelos de scikit-learn que vamos a usar aceptan solo features numéricas. Las categóricas hay que convertirlas a números. Tenemos dos tipos distintos de categóricas y merecen tratamientos distintos.

### One-hot encoding para categóricas con pocos niveles

`distrito`, `franja` y `tipo_elem` tienen pocos valores únicos — del orden de decenas como mucho. Para estas, la técnica estándar es **one-hot encoding**: crear una columna binaria por cada valor posible. Por ejemplo, si `franja` tiene 5 valores, se convierte en 5 columnas (`franja_madrugada`, `franja_mañana`, ...) donde cada fila tiene un 1 en la que corresponde y 0 en el resto.

La ventaja del one-hot es que no impone ningún orden entre las categorías. Si en lugar de one-hot usáramos una codificación entera (`madrugada=0, mañana=1, mediodia=2, ...`), el modelo podría interpretar erróneamente que "mediodia" está "a medio camino" entre "madrugada" y "tarde", lo cual no tiene sentido. Con one-hot, cada categoría es independiente de las demás.

In [ ]:
df_encoded = pd.get_dummies(
    df,
    columns=['distrito', 'franja', 'tipo_elem'],
    drop_first=False,   # Mantenemos todas las categorías. drop_first ahorra una columna pero complica la interpretación.
    dtype=int,
)

# ¿Cuántas columnas nuevas ha añadido?
nuevas = [c for c in df_encoded.columns
          if c.startswith(('distrito_', 'franja_', 'tipo_elem_'))]
print(f'Columnas one-hot creadas: {len(nuevas)}')


### Codificación entera para `id`

`id` es diferente. Hay miles de sensores distintos. Aplicar one-hot encoding crearía miles de columnas nuevas — un disparate en términos de memoria y tiempo de entrenamiento. Para variables categóricas con alta cardinalidad, lo razonable es una **codificación entera**: simplemente asignar a cada valor único un número del 0 al N-1.

Estrictamente hablando, esto introduce un orden artificial: el sensor con código 0 queda "más cerca" del 1 que del 500. Los modelos basados en árboles (Random Forest, Gradient Boosting) toleran bastante bien esta codificación porque en cada split eligen un umbral que puede partir el espacio de IDs por donde les convenga. La regresión logística sufre más — probablemente no va a sacar mucha señal del campo `id_sensor` — pero como baseline lo usamos igual y si la regresión logística queda por debajo de los árboles, una de las causas será precisamente esto.

In [ ]:
# .astype('category').cat.codes asigna un entero único a cada valor, del 0 al N-1
df_encoded['id_sensor'] = df_encoded['id'].astype('category').cat.codes

print(f'Sensores únicos codificados: {df_encoded["id_sensor"].nunique()}')


Guardamos también el mapeo inverso (código entero → id original del sensor) en un diccionario. Lo necesitaremos en el notebook 04 cuando hagamos predicciones desde la función de demo y tengamos que traducir de vuelta al id real para buscar nombre y distrito en el catálogo.

In [ ]:
categorias_sensor = df_encoded['id'].astype('category').cat.categories
mapeo_sensores    = dict(enumerate(categorias_sensor))

print(f'Primeras entradas del mapeo: {list(mapeo_sensores.items())[:3]}')


Y ahora reconstruimos la lista final de features: las numéricas originales, más la codificación entera del sensor, más todas las columnas one-hot que ha generado `get_dummies`.

In [ ]:
features_onehot = [c for c in df_encoded.columns
                   if c.startswith(('distrito_', 'franja_', 'tipo_elem_'))]
features_final  = features_numericas + ['id_sensor'] + features_onehot

print(f'Nº total de features: {len(features_final)}')
print(f'   numéricas:      {len(features_numericas)}')
print(f'   id_sensor:      1')
print(f'   one-hot:        {len(features_onehot)}')


## 8. Train / test split

Antes de entrenar modelos hay que apartar una parte de los datos que no tocaremos hasta el notebook 04. Es el pacto fundamental del machine learning aplicado: el test set es el juez, y solo se mira al final. Si lo miramos antes — aunque sea "para ver" — ya no es un juez imparcial, porque hemos usado esa información para tomar decisiones.

Usamos 80% para entrenar y 20% para test. Es la partición clásica y funciona bien cuando hay suficientes datos (tenemos millones de lecturas; con un 20% tenemos cientos de miles para evaluar, de sobra).

### ¿Split aleatorio o temporal?

Esta es una decisión importante. Hay dos filosofías.

Un **split temporal** separa los datos por fecha: por ejemplo, entrenar con los primeros 24 días del mes y testear con los últimos 6. Es la opción más honesta cuando el dataset cubre un período largo y queremos simular el despliegue en producción — donde el modelo entrenará con el pasado y predecirá el futuro.

Un **split aleatorio** mezcla todo y coge el 20% al azar. Es la opción que usamos aquí porque el dataset son sólo unas pocas semanas: con un split temporal, el test set serían solo 5-6 días, lo que genera mucha varianza dependiendo de qué días caen dentro (un festivo inesperado puede cambiar bastante los resultados).

Cuando más adelante carguemos varios meses de datos, merece la pena revisar esto y pasar a un split temporal. Por ahora, aleatorio con estratificación por el target para que la proporción de congestiones sea la misma en train y en test.

In [ ]:
X = df_encoded[features_final]
y = df_encoded[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    stratify=y,                 # Clave para que train y test tengan la misma proporción de positivos
    random_state=RANDOM_STATE,  # Reproducibilidad
)

print(f'Train: {X_train.shape[0]:>10,} registros')
print(f'Test:  {X_test.shape[0]:>10,} registros')
print(f'Proporción congestionados en train: {y_train.mean():.2%}')
print(f'Proporción congestionados en test:  {y_test.mean():.2%}')


Las dos proporciones cuadran entre sí y con la del dataset completo — señal de que la estratificación funcionó como esperábamos.

## 9. Guardado del dataset procesado

El notebook termina dejando en `data/processed/` todo lo que necesitarán los notebooks siguientes. Usamos pickle porque preserva los dtypes exactos de pandas (con CSV perderíamos, por ejemplo, el hecho de que una columna es `int` y no `float`). El fichero de sensores con nombres de calle también se guarda, porque el notebook 04 lo usará para mostrar predicciones con contexto legible.

In [ ]:
splits = {
    'X_train':  X_train,
    'X_test':   X_test,
    'y_train':  y_train,
    'y_test':   y_test,
    'features': features_final,
}

with open(DATA_PROCESSED / 'splits.pkl', 'wb') as f:
    pickle.dump(splits, f)


El DataFrame con la metadata de sensores y el mapeo inverso de ids se guardan aparte.

In [ ]:
df_sensores_mini.to_pickle(DATA_PROCESSED / 'sensores.pkl')

with open(DATA_PROCESSED / 'mapeo_sensores.pkl', 'wb') as f:
    pickle.dump(mapeo_sensores, f)


Listamos lo que ha quedado en disco para dejar constancia.

In [ ]:
print('Ficheros guardados en', DATA_PROCESSED.resolve())
for p in sorted(DATA_PROCESSED.glob('*.pkl')):
    print(f'  {p.name:25s} {p.stat().st_size / 1024:>8.1f} KB')


## 10. Resumen

Partimos de unos CSV con errores y los hemos transformado en un dataset listo para entrenar. Las decisiones importantes a recordar son las siguientes.

Quitamos los registros con cualquiera de los dos marcadores de error (carga = -1 o error = 'S') en lugar de imputarlos, porque no tenemos base para reconstruir esos valores y porque tenemos de sobra sin ellos. Construimos features temporales básicas (hora, día de la semana, mes) y derivadas (es laborable, es hora punta, franja) que capturan los patrones que el EDA hizo visibles. Fijamos el umbral de congestión en 70% — un desbalance moderado, manejable con class_weight. Mantuvimos el sensor como variable de entrada (codificado como entero por alta cardinalidad) para que el modelo pueda aprender comportamientos específicos de cada punto. Dejamos fuera cualquier variable que se mida en el mismo instante que la carga, evitando data leakage. Partimos los datos 80/20 con estratificación para que el test mantenga la proporción de positivos. Y dejamos todo en pickle en `data/processed/` para el resto del pipeline.

El notebook 03 recoge `splits.pkl` y entrena los modelos.